# Unsupervised Learning

## Necessary Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio; pio.renderers.default = "notebook"
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## Dataset Load

In [2]:
all_players = pd.read_csv("../Data/Clean/All_Players.csv")
all_players_2024 = all_players[all_players["season"] == 2024].copy()

Note: Only the 2024 season is analyzed here. This way, each hitter appears at most once, so the clusters group different hitters rather than several seasons of the same hitter, and the PCA visualization isn't crowded with repeated players.

## Standardization

Before applying $K$-means clustering or PCA, each feature is standardized. The features in this dataset are measured in very different units (e.g., miles per hour, feet, counts, and proportions), so their scales differ by orders of magnitude. Since $K$-means relies on Euclidean distances and PCA relies on variances and covariances, features with large numerical ranges, such as `maximum_distance_hit`, would otherwise dominate the results simply because of their units, not because they are more important.

For the $i\text{th}$ feature, each observation $x_{ni}$ is replaced by its $Z$-score:

$$
\tilde{x}_{ni}=\frac{x_{ni}-\overline{x}_i}{s_i} \tag*{(1)}
$$

where

$$
\overline{x}_i=\frac{1}{N}\sum_{n=1}^Nx_{ni},\qquad s_i=\sqrt{\frac{1}{N}\sum_{n=1}^N(x_{ni}-\overline{x}_i)^2} \tag*{(2)}
$$

so that every standardized feature has mean $0$ and standard deviation $1$. This is exactly what `StandardScaler` does. Note that it uses the $1/N$ (population) standard deviation rather than the $1/(N-1)$ sample standard deviation, which is equivalent to `np.std` with `ddof = 0`.

Let $X_i$ denote the $i\text{th}$ original feature and $\tilde{X}_i$ its standardized version. A useful consequence is that the covariance between two standardized features equals the correlation between the original features:

$$
\mathrm{Cov}\left(\tilde{X}_i,\,\tilde{X}_j\right)=\frac{\mathrm{Cov}(X_i,\,X_j)}{s_i\,s_j}=r_{ij} \tag*{(3)}
$$

Since $r_{ij}$ is a ratio, the choice of $1/N$ or $1/(N-1)$ cancels (as long as the covariance and standard deviations use the same convention), so $r_{ij}$ is the same Pearson correlation coefficient used in the EDA. Hence, PCA on standardized data is equivalent to PCA on the correlation matrix of the original features.

## $K$-means Clustering

The primary goal of $K$-means clustering is to minimize the inertia metric: the total within-cluster sum of squares.

Let a given dataset consist of $N$ observations $\vec{x}_1,\,\vec{x}_2,\,\ldots,\,\vec{x}_N\in\mathbb{R}^p$ (here, the standardized observations) where each observation has $p$ features.

If the data are partitioned into $K$ clusters $C_1,\,C_2,\,\ldots,\,C_K$ with corresponding centroids $\vec{\alpha}_1,\,\vec{\alpha}_2,\,\ldots,\,\vec{\alpha}_K\in\mathbb{R}^p$, then the inertia is 

$$
\iota=\sum_{k=1}^K\sum_{\vec{x}_n\in C_k}\|\vec{x}_n-\vec{\alpha}_k\|^2 \tag*{(4)}
$$

Starting from $K$ initial centroids, the $K$-means algorithm (Lloyd's algorithm) alternates between two steps until the clusters stop changing:

1. Assignment step: each observation is assigned to the cluster with the nearest centroid (ties broken arbitrarily):

$$
C_k=\left\{\vec{x}_n:\|\vec{x}_n-\vec{\alpha}_k\|^2\leq\|\vec{x}_n-\vec{\alpha}_j\|^2\text{ for all }j\right\} \tag*{(5)}
$$

2. Update step: each centroid is replaced by the mean of the observations in its cluster. To see why, note that with the clusters fixed, $\iota$ splits into $K$ separate terms, one for each centroid. Setting the gradient of the $k\text{th}$ term with respect to $\vec{\alpha}_k$ equal to zero gives

$$
\nabla_{\vec{\alpha}_k}\sum_{\vec{x}_n\in C_k}\|\vec{x}_n-\vec{\alpha}_k\|^2=-2\sum_{\vec{x}_n\in C_k}(\vec{x}_n-\vec{\alpha}_k)=\vec{0}\implies\vec{\alpha}_k=\frac{1}{|C_k|}\sum_{\vec{x}_n\in C_k}\vec{x}_n \tag*{(6)}
$$

and since each term is a convex quadratic in $\vec{\alpha}_k$, this critical point is its minimum.

Neither step can increase $\iota$: the assignment step moves each observation to its nearest centroid, and by $(6)$, the update step moves each centroid to the point that minimizes its cluster's sum of squares. Since $\iota$ never increases and there are only finitely many ways to partition $N$ observations into $K$ clusters, the algorithm must eventually reach a partition that no longer changes, so it is guaranteed to converge. However, it converges to a local minimum, not necessarily the global minimum, and the result depends on where the centroids are initialized.

For this reason, `sklearn`'s `KMeans` uses $K$-means++ initialization by default, which spreads the initial centroids apart: the first centroid is chosen uniformly at random from the observations, and each subsequent centroid is chosen from the observations with probability proportional to its squared distance from the nearest centroid already chosen. Since `sklearn` 1.4, the default `n_init = "auto"` runs the algorithm only once when $K$-means++ is used. To run it from several initializations and keep the result with the lowest inertia, `n_init` must be set explicitly (e.g., `n_init = 10`).

### The Silhouette Score

For a clustering algorithm, the silhouette score at a point $\vec{x}_n$ can be used to evaluate the clustering result and help determine the best choice of $K$ (a hyperparameter).

If $\vec{x}_n\in C_k$, then the silhouette score is 

$$
s_n=\frac{b_n-a_n}{\max\{a_n,\,b_n\}} \tag*{(7)}
$$

where

$$
a_n=\frac{1}{|C_k|-1}\sum_{\vec{x}_m\in C_k,\,m\neq n}d(\vec{x}_m,\,\vec{x}_n) \tag*{(8)}
$$

is the average distance from $\vec{x}_n$ to all other points in its own cluster, using the Euclidean distance $d$ defined in $(11)$ below, and

$$
b_n=\min_{j\neq k}\left\{\frac{1}{|C_j|}\sum_{\vec{x}_m\in C_j}d(\vec{x}_m,\,\vec{x}_n)\right\} \tag*{(9)}
$$

is the smallest average distance from $\vec{x}_n$ to points in another cluster.

The overall silhouette score $\varphi$ for the clustering is the mean of all pointwise scores $s_n$:

$$
\varphi=\frac{1}{N}\sum_{n=1}^Ns_n \tag*{(10)}
$$

Finally, values of $s_n$ close to $1$ indicate well-separated clusters, values near $0$ indicate overlapping clusters, and negative values suggest possible misclassifications.

Note: The Euclidean distance between two points $\vec{u},\,\vec{v}\in\mathbb{R}^p$ can be described as the function $d\colon\mathbb{R}^p\times\mathbb{R}^p\to\mathbb{R}$ where

$$
d(\vec{u},\,\vec{v})=\sqrt{\sum_{i=1}^p(v_i-u_i)^2}=\|\vec{v}-\vec{u}\| \tag*{(11)}
$$

Also, $a_n$ is undefined when a point $\vec{x}_n$ is alone in its cluster $C_k$, as $|C_k|-1$ is the denominator in the formula for $a_n$. `sklearn` handles this edge case by setting $s_n=0$. For similar reasons, the silhouette score requires $2\leq K\leq N-1$. If $K=1$, there is no other cluster, so $b_n$ is undefined. If $K=N$, every point is alone in its own cluster, so every $s_n=0$ and the score carries no information.

## Principal Component Analysis

Principal component analysis (PCA) is a dimensionality reduction technique used to visualize high-dimensional data by constructing the $2$ or $3$ most significant principal components. 

These principal components are orthogonal linear combinations of the original features. Their directions are the eigenvectors of the covariance matrix associated with the largest eigenvalues, which point in the directions of maximum variance in the data.

After standardizing each feature (see the Standardization section above), the covariance matrix for a $p$-dimensional feature space is given by

$$
\Psi 
=
\left(
    \begin{array}{cccc}
        \mathrm{Var}(\tilde{X}_1) & \mathrm{Cov}(\tilde{X}_1,\,\tilde{X}_2) & \cdots & \mathrm{Cov}(\tilde{X}_1,\,\tilde{X}_p) \\
        \mathrm{Cov}(\tilde{X}_2,\,\tilde{X}_1) & \mathrm{Var}(\tilde{X}_2) & \cdots & \mathrm{Cov}(\tilde{X}_2,\,\tilde{X}_p) \\
        \vdots & \vdots & \ddots & \vdots \\
        \mathrm{Cov}(\tilde{X}_p,\,\tilde{X}_1) & \mathrm{Cov}(\tilde{X}_p,\,\tilde{X}_2) & \cdots & \mathrm{Var}(\tilde{X}_p)
    \end{array}
\right)
\in
\mathbb{R}^{p\times p}
\tag*{(12)}
$$

Since each standardized feature $\tilde{X}_i$ has variance $1$, every diagonal entry of $\Psi$ equals $1$, and by $(3)$, each off-diagonal entry is the correlation $r_{ij}$ between the original features $X_i$ and $X_j$. In other words, $\Psi$ is the correlation matrix.

To find the eigenvalues $\lambda_i$ of $\Psi$ for $1\leq i\leq p$, one solves the following, where $\mathcal{I}\in\mathbb{R}^{p\times p}$ is the identity matrix:

$$
\det\left(\Psi-\lambda\mathcal{I}\right)=0 \tag*{(13)}
$$

Since $\Psi$ is symmetric, its eigenvalues are real, and since $\vec{v}^\top\Psi\,\vec{v}$ is the variance of the data projected onto a unit vector $\vec{v}$, which can never be negative, its eigenvalues are also nonnegative $(\lambda_i\geq0)$. In fact, $\lambda_i$ is exactly the variance of the data along the $i\text{th}$ principal component.

Ordering the eigenvalues so that $\lambda_1\geq\lambda_2\geq\cdots\geq\lambda_p$, the first principal component corresponds to $\lambda_1$, the second to $\lambda_2$, and so on.

Then, to find the eigenvector $\vec{v}_i\in\mathbb{R}^p$ that corresponds to each eigenvalue $\lambda_i$, one solves the following system of equations for the components of $\vec{v}_i$:

$$
\Psi\,\vec{v}_i=\lambda_i\,\vec{v}_i \tag*{(14)}
$$

In the $2$-dimensional case, for some $\lambda\in\mathbb{R}$ that corresponds to $\vec{v}\in\mathbb{R}^2$, this system looks like

$$
\begin{cases}
    \psi_{11}v_1+\psi_{12}v_2=\lambda v_1 \\
    \psi_{21}v_1+\psi_{22}v_2=\lambda v_2
\end{cases}
\tag*{(15)}
$$

where $\Psi\in\mathbb{R}^{2\times2}$ and $\psi_{ij}$ are the entries of $\Psi$.

To decide how many principal components to keep, one can compute the proportion of the total variance explained by the $i\text{th}$ principal component, called its explained-variance ratio:

$$
\frac{\lambda_i}{\sum_{j=1}^p\lambda_j}=\frac{\lambda_i}{\mathrm{tr}(\Psi)} \tag*{(16)}
$$

where $\mathrm{tr}(\Psi)$ is the trace of $\Psi$, the sum of its diagonal entries. The equality holds because the trace of a square matrix always equals the sum of its eigenvalues. Since every diagonal entry of $\Psi$ is $1$, $\mathrm{tr}(\Psi)=p$. Keeping $2$ or $3$ components is well justified when their explained-variance ratios add up to a large share of the total. These values are available in `sklearn` as the `explained_variance_ratio_` attribute of a fitted `PCA` object.

In practice, `sklearn`'s `PCA` does not solve $(13)$ directly. Instead, it computes the singular value decomposition (SVD) of the standardized data matrix $\tilde{X}\in\mathbb{R}^{N\times p}$, whose $i\text{th}$ column is $\tilde{X}_i$:

$$
\tilde{X}=U\Delta V^\top \tag*{(17)}
$$

where $\Delta$ is a diagonal matrix of singular values $\delta_1\geq\delta_2\geq\cdots\geq\delta_p\geq0$. The columns of $V$ are the eigenvectors $\vec{v}_i$, and the eigenvalues are recovered as $\lambda_i=\delta_i^2/(N-1)$. Note that `sklearn` divides by $N-1$ here, while `StandardScaler` uses $1/N$, so the eigenvalues it reports are $N/(N-1)$ times those of the correlation matrix and sum to $pN/(N-1)$ rather than exactly $p$. The explained-variance ratios in $(16)$ are unaffected, since the factor cancels.

Finally, the data are projected onto the leading $q$ eigenvectors to obtain lower-dimensional coordinates for visualization and analysis:

$$
T=\tilde{X}V_q\in\mathbb{R}^{N\times q} \tag*{(18)}
$$

where $V_q\in\mathbb{R}^{p\times q}$ has columns $\vec{v}_1,\,\ldots,\,\vec{v}_q$, and the $n\text{th}$ row of $T$ gives the coordinates of the $n\text{th}$ observation along the first $q$ principal components.

Note: The eigenvectors are normalized $(\|\vec{v}_i\|=1)$, as otherwise the projections aren't uniquely scaled. Even then, each eigenvector is only unique up to its sign, since $-\vec{v}_i$ is also a unit eigenvector for $\lambda_i$, so a principal component's axis may appear flipped across different software or runs.